# 🏢 Chapter 03: Building Morphometrics & In-Database DBSCAN Clustering
## Distinguishing Informal Settlements from Planned Neighborhoods

### 🎯 The Mission: Why We Are Doing This
In Greater Accra, urban morphology reflects socioeconomic settlement patterns:
- **Planned enclaves (Cantonments, Airport Residential):** Large footprint areas ($A > 300	ext{ m}^2$), high compactness ($CI > 0.75$), regular spacing.
- **Informal / Organic settlements (Nima, Old Fadama, Jamestown):** Small footprints ($A < 80	ext{ m}^2$), lower compactness, high spatial density forming massive contiguous clusters.

Using PostGIS computational geometry functions and in-database **DBSCAN Clustering (`ST_ClusterDBSCAN`)**, we classify building typologies directly inside SQL.

---

### 🔑 Spatial DBA Keywords Decoded

| Keyword / Metric | Plain-English Explanation | Production Purpose |
| :--- | :--- | :--- |
| **Isoperimetric Quotient ($CI$)** | Compactness formula: $CI = rac{4\pi A}{P^2}$. Circles = $1.0$, rectangles $pprox 0.78$, fractured shapes $< 0.45$. | Mathematical quantifier of structural irregularity. |
| **`ST_OrientedEnvelope(geom)`** | Computes the smallest rotated bounding rectangle enclosing the polygon. | Used to measure true length, width, and rectangular fit. |
| **`ST_ClusterDBSCAN(geom, eps, minpts)`** | Groups spatial objects into clusters if at least `minpts` features lie within metric distance `eps`. | Automatically detects contiguous informal settlement clusters. |

In [ ]:
from config_db import execute_sql, query_df, query_gdf, preview_map

sql_morpho = """
DROP TABLE IF EXISTS accra_building_inventory CASCADE;
CREATE TABLE accra_building_inventory (
    building_id SERIAL PRIMARY KEY,
    neighborhood VARCHAR(50),
    geom GEOMETRY(Polygon, 32630)
);

INSERT INTO accra_building_inventory (neighborhood, geom)
VALUES 
    ('Cantonments', ST_GeomFromText('POLYGON((810000 615000, 810030 615000, 810030 615025, 810000 615025, 810000 615000))', 32630)),
    ('Cantonments', ST_GeomFromText('POLYGON((810060 615000, 810090 615000, 810090 615025, 810060 615025, 810060 615000))', 32630)),
    ('Cantonments', ST_GeomFromText('POLYGON((810000 615060, 810030 615060, 810030 615085, 810000 615085, 810000 615060))', 32630)),
    ('Nima', ST_GeomFromText('POLYGON((808000 616000, 808010 616000, 808010 616008, 808000 616008, 808000 616000))', 32630)),
    ('Nima', ST_GeomFromText('POLYGON((808012 616000, 808022 616000, 808022 616008, 808012 616008, 808012 616000))', 32630)),
    ('Nima', ST_GeomFromText('POLYGON((808000 616010, 808010 616010, 808010 616018, 808000 616018, 808000 616010))', 32630)),
    ('Nima', ST_GeomFromText('POLYGON((808012 616010, 808022 616010, 808022 616018, 808012 616018, 808012 616010))', 32630)),
    ('Nima', ST_GeomFromText('POLYGON((808024 616000, 808034 616000, 808034 616018, 808024 616018, 808024 616000))', 32630));

DROP TABLE IF EXISTS accra_morphometric_analysis CASCADE;
CREATE TABLE accra_morphometric_analysis AS
WITH computed_metrics AS (
    SELECT 
        building_id, neighborhood, geom,
        ST_Area(geom) AS footprint_area_m2,
        ST_Perimeter(geom) AS perimeter_m,
        (4 * PI() * ST_Area(geom)) / NULLIF(POWER(ST_Perimeter(geom), 2), 0) AS compactness_ci,
        ST_Area(geom) / NULLIF(ST_Area(ST_OrientedEnvelope(geom)), 0) AS rectangular_fit
    FROM accra_building_inventory
)
SELECT 
    building_id, neighborhood, geom,
    ROUND(footprint_area_m2::NUMERIC, 1) AS area_m2,
    ROUND(perimeter_m::NUMERIC, 1) AS perimeter_m,
    ROUND(compactness_ci::NUMERIC, 3) AS compactness_ci,
    ROUND(rectangular_fit::NUMERIC, 3) AS rectangular_fit,
    CASE 
        WHEN compactness_ci < 0.60 AND footprint_area_m2 < 100 THEN 'Organic / High Density'
        ELSE 'Formal / Planned'
    END AS morphological_class,
    ST_ClusterDBSCAN(geom, eps := 15, minpoints := 3) OVER () AS cluster_group_id
FROM computed_metrics;

CREATE INDEX idx_morpho_geom ON accra_morphometric_analysis USING GIST(geom);
"""

execute_sql(sql_morpho)
print("✅ Morphometrics and DBSCAN clustering completed.")

In [ ]:
# View Morphometrics Summary
df_morpho = query_df("""
SELECT neighborhood, morphological_class, COUNT(*) AS count, ROUND(AVG(area_m2)::NUMERIC, 1) AS avg_area_m2, ROUND(AVG(compactness_ci)::NUMERIC, 3) AS avg_compactness
FROM accra_morphometric_analysis
GROUP BY neighborhood, morphological_class;
""")

df_morpho

In [ ]:
# Interactive Map Preview (Color by Cluster ID)
gdf_morpho = query_gdf("""
SELECT building_id, neighborhood, morphological_class, cluster_group_id, geom FROM accra_morphometric_analysis;
""")

preview_map(gdf_morpho, tooltip_cols=["building_id", "neighborhood", "morphological_class", "cluster_group_id"])